# Single-Agent Smart Assistant — Agent Pipeline Project

## Problem Statement
This notebook builds a **single-agent smart assistant** that can understand a user's query, decide which tool (if any) fits the request, run that tool, and always return a clean, structured response — regardless of what path it took internally.

The agent supports three intents:
- **Math queries** → routed to a Calculator Tool
- **Keyword extraction requests** → routed to a Keyword Extractor Tool
- **Everything else** → handled as a general/direct response

### What's implemented here
- Two core tools (Calculator, Keyword Extractor) plus one bonus tool (Word Counter)
- Conditional routing logic in the agent function
- A lightweight **state log** that tracks which "node" (step) the agent visited for each query — this reflects the stateful directed graph idea from the quiz (nodes = steps, edges = the routing decisions between them)
- A schema check so every response follows the same `{"type": ..., "result": ...}` structure
- Error handling with try/except around every tool call
- Test cases + an interactive mode to try it live


## Tool 1 — Calculator

Rather than using raw `eval()` on the whole expression (which can break on stray words and is unsafe for arbitrary input), this version:
1. Strips out anything that isn't a digit, operator, decimal point, or parenthesis
2. Evaluates only the cleaned numeric expression

This makes the tool a bit more robust to messy input like `"calculate 20 + 5 please"`.


In [1]:
import re

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely.

    Strips non-math characters first so trigger words like 'calculate'
    or trailing punctuation don't break the evaluation.
    """
    try:
        # Keep only digits, operators (incl. modulo %), decimal points, spaces, and parentheses
        cleaned = re.sub(r"[^0-9+\-*/%.() ]", "", expression)
        cleaned = cleaned.strip()
        if not cleaned:
            return "Error in calculation"
        return str(eval(cleaned, {"__builtins__": {}}, {}))
    except Exception:
        return "Error in calculation"

## Tool 2 — Keyword Extractor

Improved slightly over a plain "any word longer than 4 letters" filter by ignoring a small
set of common stopwords, so filler words like "about" or "which" don't crowd out real keywords.


In [2]:
STOPWORDS = {"about", "which", "there", "their", "would", "these", "those", "extract", "keywords", "from"}

def extract_keywords(text: str) -> list:
    """Extract up to 5 keywords from text, ignoring common stopwords."""
    try:
        words = text.split()
        keywords = [w.lower().strip(".,!?") for w in words if len(w) > 4]
        keywords = [w for w in keywords if w not in STOPWORDS]
        # de-duplicate while preserving order
        seen = set()
        unique_keywords = []
        for w in keywords:
            if w not in seen:
                seen.add(w)
                unique_keywords.append(w)
        return unique_keywords[:5]
    except Exception:
        return []


## Bonus Tool — Word Counter

A small third tool to demonstrate the agent can scale to more than two intents without
changing its core structure — just one more routing branch and one more node.


In [3]:
def word_counter(text: str) -> int:
    """Count the number of words in the given text."""
    try:
        return len(text.split())
    except Exception:
        return 0


## Agent Logic — Routing, State, and Schema Validation

The `agent()` function is the core of this project. It does four things every time it runs:

1. **Routes** the query to the right tool based on keywords in the text (conditional routing)
2. **Calls** that tool, wrapped in error handling so a tool failure never crashes the agent
3. **Logs** which node it visited into `execution_log` — a simple stand-in for the "state" that
   would be tracked across a stateful directed graph
4. **Validates** the output against the expected schema before returning it, so the response
   shape is always `{"type": ..., "result": ...}`


In [4]:
execution_log = []  # tracks the path the agent has taken across all queries so far

def looks_like_math(query: str) -> bool:
    """Check if a query is purely a math expression (digits/operators only),
    even if it doesn't contain the word 'calculate'.
    """
    stripped = query.strip().strip('"').strip("'")
    return bool(re.fullmatch(r"[0-9+\-*/%.() ]+", stripped)) and stripped != ""


def validate_schema(response: dict) -> bool:
    """Confirm the response matches the expected {type, result} schema."""
    return isinstance(response, dict) and "type" in response and "result" in response


def agent(query: str) -> dict:
    query_lower = query.lower()
    node_visited = None

    try:
        if "calculate" in query_lower or looks_like_math(query):
            node_visited = "calculator_tool"
            expression = query_lower.replace("calculate", "").strip().strip('"').strip("'")
            result = calculator(expression)
            response = (
                {"type": "error", "result": result}
                if result == "Error in calculation"
                else {"type": "calculation", "result": result}
            )

        elif "keywords" in query_lower:
            node_visited = "keyword_tool"
            text = query_lower.replace("extract keywords from", "").strip()
            response = {"type": "keywords", "result": extract_keywords(text)}

        elif "count words" in query_lower or "word count" in query_lower:
            node_visited = "word_counter_tool"
            text = query_lower.replace("count words in", "").replace("word count of", "").strip()
            response = {"type": "word_count", "result": word_counter(text)}

        else:
            node_visited = "general_response"
            response = {
                "type": "general",
                "result": f"You asked: '{query}'. This is a general query with no specific tool.",
            }

    except Exception as e:
        node_visited = "error_handler"
        response = {"type": "error", "result": f"Something went wrong: {str(e)}"}

    execution_log.append({"query": query, "node": node_visited})

    if not validate_schema(response):
        response = {"type": "error", "result": "Response did not match expected schema"}

    return response

## Expected Output Format

Every call to `agent()` returns:
```
{
  "type": "calculation / keywords / word_count / general / error",
  "result": ...
}
```


## Test Cases

In [5]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Count words in The quick brown fox jumps over the lazy dog",
    "What is machine learning?",
    '"50+6"',
    '"50+6*7%10-5"',
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
--------------------------------------------------
Query: Count words in The quick brown fox jumps over the lazy dog
Response: {'type': 'word_count', 'result': 9}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "You asked: 'What is machine learning?'. This is a general query with no specific tool."}
--------------------------------------------------
Query: "50+6"
Response: {'type': 'calculation', 'result': '56'}
--------------------------------------------------
Query: "50+6*7%10-5"
Response: {'type': 'calculation', 'result': '47'}
--------------------------------------------------


## Viewing the Agent's State (Execution Log)

This prints the sequence of nodes visited across all queries run so far — a small,
concrete example of state being carried and accumulated across calls, tying back to
the stateful directed graph concept from the quiz.


In [ ]:
for entry in execution_log:
    print(entry)


## Interactive Mode

Run this cell to try your own queries. Type `exit` to stop.

In [6]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))


Enter query (type 'exit' to stop): 20 * 5
Response: {'type': 'calculation', 'result': '100'}
Enter query (type 'exit' to stop): what is Artificial Intelligrnce? 
Response: {'type': 'general', 'result': "You asked: 'what is Artificial Intelligrnce? '. This is a general query with no specific tool."}
Enter query (type 'exit' to stop): 70*9+6%10
Response: {'type': 'calculation', 'result': '636'}
Enter query (type 'exit' to stop): extract keywords from celebal technologies 
Response: {'type': 'keywords', 'result': ['celebal', 'technologies']}
Enter query (type 'exit' to stop): exit
